<a href="https://colab.research.google.com/github/shreeraj-chatterjee/financial-fraud-classification/blob/main/gnn_fraud_detection_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Check if GPU is available

In [ ]:
import torch
print(torch.cuda.is_available())

Define the device (Output should be: cuda)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Run this once per session (Access to my Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1: Install Dependencies
# We can rely on Colab's pre-installed PyTorch and just install torch-geometric directly.
!pip install torch-geometric scikit-learn matplotlib pandas

In [ ]:
# Cell 2: Data Loading and Preprocessing
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets/creditcard.csv')

print("Original Dataset Shape:", df.shape)
print("Fraudulent transactions:", len(df[df['Class'] == 1]))
print("Legitimate transactions:", len(df[df['Class'] == 0]))

# The dataset is highly imbalanced. Let's balance it using undersampling
# to ensure our Graph fits in Google Colab's RAM.
fraud_df = df[df['Class'] == 1]
legit_df = df[df['Class'] == 0].sample(n=len(fraud_df)*5, random_state=42) # Keep 5x more legit transactions

balanced_df = pd.concat([fraud_df, legit_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced Dataset Shape:", balanced_df.shape)

# Separate features and labels
# We drop 'Time', 'Class' as they aren't predictive features in the same way
features = balanced_df.drop(columns=['Time', 'Class']).values
labels = balanced_df['Class'].values

# Standardize features (Mean = 0, Variance = 1)
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Convert to PyTorch tensors
X = torch.tensor(features, dtype=torch.float32).to(device)
y = torch.tensor(labels, dtype=torch.long).to(device)

print("Feature Tensor Shape:", X.shape)
print("Label Tensor Shape:", y.shape)

In [ ]:
# Cell 3: Graph Construction
from sklearn.neighbors import kneighbors_graph
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
import numpy as np

def create_knn_edge_index(X_np, k=5):
    print(f"Building KNN graph with k={k}...")

    knn_graph = kneighbors_graph(
        X_np,
        n_neighbors=k,
        mode='connectivity',
        include_self=False,
        n_jobs=-1
    )

    rows, cols = knn_graph.nonzero()
    edge_index = torch.tensor(np.vstack((rows, cols)), dtype=torch.long)
    edge_index = to_undirected(edge_index).to(device)

    return edge_index

# Create the edge index using the numpy version of our features
edge_index = create_knn_edge_index(features, k=5)

# Create the PyTorch Geometric Data object
data = Data(x=X, edge_index=edge_index, y=y)

print("\nGraph Conversion Successful!")
print(f"Number of Nodes (Transactions): {data.num_nodes}")
print(f"Number of Edges (Connections): {data.num_edges}")

In [ ]:
# Cell 4: Define and Initialize the GNN Model

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv

class GNN(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features):
        super(GNN, self).__init__()
        # First Graph Convolutional Layer
        self.conv1 = GCNConv(in_features, hidden_dim)
        # Second Graph Convolutional Layer
        self.conv2 = GCNConv(hidden_dim, out_features)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Pass data through the first layer
        x = self.conv1(x, edge_index)
        # Apply ReLU activation function to learn non-linear patterns
        x = F.relu(x)

        # Pass data through the second layer
        x = self.conv2(x, edge_index)

        # Use log_softmax for our output probabilities
        return F.log_softmax(x, dim=1)

# Set the dimensions
num_features = X.shape[1] # Number of input features from our dataset
hidden_size = 16          # Number of hidden neurons (can be tuned later)
num_classes = 2           # 0 for Legitimate, 1 for Fraudulent

# Initialize the model
model = GNN(in_features=num_features, hidden_dim=hidden_size, out_features=num_classes)

# Move the model to the GPU!
model = model.to(device)

# Define the Optimizer (Adam) and the Loss Function (Cross Entropy)
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

print("Model Architecture:")
print(model)
print(f"Model successfully moved to: {device}")

In [ ]:
# Cell 5: Training the GNN and Evaluating Performance

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt

# 1. Create Train/Test Masks
# We split the indices: 80% for training, 20% for testing
train_indices, test_indices = train_test_split(
    np.arange(data.num_nodes),
    test_size=0.2,
    random_state=42,
    stratify=data.y.cpu().numpy() # Ensure equal ratio of fraud in both sets
)

# Convert masks to PyTorch tensors and move them to the GPU
train_mask = torch.tensor(train_indices, dtype=torch.long).to(device)
test_mask = torch.tensor(test_indices, dtype=torch.long).to(device)

# 2. Train the Model
epochs = 50
train_losses = []
test_losses = []

print("Starting Training...")
for epoch in range(epochs):
    model.train()           # Set model to training mode
    optimizer.zero_grad()   # Clear old gradients

    out = model(data)       # Forward pass: get predictions for all nodes

    # Calculate loss ONLY on the training nodes
    loss = criterion(out[train_mask], data.y[train_mask])
    loss.backward()         # Backward pass: calculate gradients
    optimizer.step()        # Update weights

    train_losses.append(loss.item())

    # Evaluate on the test set
    model.eval()            # Set model to evaluation mode
    with torch.no_grad():   # Turn off gradient tracking to save memory
        test_out = model(data)
        test_loss = criterion(test_out[test_mask], data.y[test_mask])
        test_losses.append(test_loss.item())

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

# 3. Final Evaluation Metrics
model.eval()
with torch.no_grad():
    preds = model(data)
    # Since we used log_softmax in our model, we use torch.exp to get actual probabilities
    y_prob = torch.exp(preds[:, 1])
    y_pred = preds.argmax(dim=1)

# Move predictions and true labels back to the CPU for sklearn calculations
y_true_cpu = data.y[test_mask].cpu().numpy()
y_pred_cpu = y_pred[test_mask].cpu().numpy()
y_prob_cpu = y_prob[test_mask].cpu().numpy()

# Calculate Metrics
accuracy = accuracy_score(y_true_cpu, y_pred_cpu)
precision = precision_score(y_true_cpu, y_pred_cpu, zero_division=1)
recall = recall_score(y_true_cpu, y_pred_cpu, zero_division=1)
f1 = f1_score(y_true_cpu, y_pred_cpu, zero_division=1)
roc_auc = roc_auc_score(y_true_cpu, y_prob_cpu)

print(f"\n--- Final Test Set Results ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC-ROC Score: {roc_auc:.4f}")

# 4. Plotting the Results
plt.figure(figsize=(12, 5))

# Plot 1: Training & Test Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, epochs+1), test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training & Test Loss')
plt.legend()

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_true_cpu, y_prob_cpu)
plt.subplot(1, 2, 2)
plt.plot(fpr, tpr, color="blue", lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Save and Download the Model Checkpoint
import torch
from google.colab import files

# Define the file name
model_save_path = 'gnn_fraud_model.pth'

# Save the model's learned weights (state dictionary)
torch.save(model.state_dict(), model_save_path)
print(f"Model successfully saved to Colab environment as: {model_save_path}")

# Download the file to your local computer for permanent safekeeping
print("Downloading to your local machine...")
files.download(model_save_path)